In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('all.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36707 entries, 0 to 36706
Data columns (total 80 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Querylength                      36707 non-null  int64  
 1   domain_token_count               36707 non-null  int64  
 2   path_token_count                 36707 non-null  int64  
 3   avgdomaintokenlen                36707 non-null  float64
 4   longdomaintokenlen               36707 non-null  int64  
 5   avgpathtokenlen                  36427 non-null  float64
 6   tld                              36707 non-null  int64  
 7   charcompvowels                   36707 non-null  int64  
 8   charcompace                      36707 non-null  int64  
 9   ldl_url                          36707 non-null  int64  
 10  ldl_domain                       36707 non-null  int64  
 11  ldl_path                         36707 non-null  int64  
 12  ldl_filename      

In [4]:
df.columns

Index(['Querylength', 'domain_token_count', 'path_token_count',
       'avgdomaintokenlen', 'longdomaintokenlen', 'avgpathtokenlen', 'tld',
       'charcompvowels', 'charcompace', 'ldl_url', 'ldl_domain', 'ldl_path',
       'ldl_filename', 'ldl_getArg', 'dld_url', 'dld_domain', 'dld_path',
       'dld_filename', 'dld_getArg', 'urlLen', 'domainlength', 'pathLength',
       'subDirLen', 'fileNameLen', 'this.fileExtLen', 'ArgLen', 'pathurlRatio',
       'ArgUrlRatio', 'argDomanRatio', 'domainUrlRatio', 'pathDomainRatio',
       'argPathRatio', 'executable', 'isPortEighty', 'NumberofDotsinURL',
       'ISIpAddressInDomainName', 'CharacterContinuityRate',
       'LongestVariableValue', 'URL_DigitCount', 'host_DigitCount',
       'Directory_DigitCount', 'File_name_DigitCount', 'Extension_DigitCount',
       'Query_DigitCount', 'URL_Letter_Count', 'host_letter_count',
       'Directory_LetterCount', 'Filename_LetterCount',
       'Extension_LetterCount', 'Query_LetterCount', 'LongestPathToken

In [5]:
select_col = [
    'Entropy_Domain', 'Entropy_DirectoryName', 'Entropy_Filename', 'Entropy_Extension',
       'Entropy_Afterpath', 'URL_Type_obf_Type'
]

In [6]:
select_df = df[select_col]

In [7]:
# -1 값의 개수를 컬럼별로 계산
minus_one_counts = (select_df == -1).sum()

In [8]:
minus_one_counts

Entropy_Domain               0
Entropy_DirectoryName     1852
Entropy_Filename          1852
Entropy_Extension         1852
Entropy_Afterpath        20481
URL_Type_obf_Type            0
dtype: int64

In [9]:
select_df.isnull().sum()

Entropy_Domain              0
Entropy_DirectoryName    8468
Entropy_Filename          236
Entropy_Extension          40
Entropy_Afterpath           6
URL_Type_obf_Type           0
dtype: int64

In [10]:
df_cleaned = select_df.dropna()

In [11]:
df_cleaned.isnull().sum()

Entropy_Domain           0
Entropy_DirectoryName    0
Entropy_Filename         0
Entropy_Extension        0
Entropy_Afterpath        0
URL_Type_obf_Type        0
dtype: int64

In [12]:
# -1이 있는 행 삭제
df_cleaned = df_cleaned[~(df_cleaned == -1).any(axis=1)]

In [13]:
(df_cleaned == -1).sum()

Entropy_Domain           0
Entropy_DirectoryName    0
Entropy_Filename         0
Entropy_Extension        0
Entropy_Afterpath        0
URL_Type_obf_Type        0
dtype: int64

In [14]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9934 entries, 37 to 36706
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Entropy_Domain         9934 non-null   float64
 1   Entropy_DirectoryName  9934 non-null   float64
 2   Entropy_Filename       9934 non-null   float64
 3   Entropy_Extension      9934 non-null   float64
 4   Entropy_Afterpath      9934 non-null   float64
 5   URL_Type_obf_Type      9934 non-null   object 
dtypes: float64(5), object(1)
memory usage: 543.3+ KB


In [15]:
df_cleaned['URL_Type_obf_Type'].value_counts()

URL_Type_obf_Type
spam          4213
malware       2400
Defacement    1458
benign        1105
phishing       758
Name: count, dtype: int64

In [16]:
df_cleaned = df_cleaned[df_cleaned['URL_Type_obf_Type'] != 'benign']

In [17]:
df_cleaned['URL_Type_obf_Type'].value_counts()

URL_Type_obf_Type
spam          4213
malware       2400
Defacement    1458
phishing       758
Name: count, dtype: int64

In [18]:
X_col = [
    'Entropy_Domain', 'Entropy_DirectoryName', 'Entropy_Filename', 'Entropy_Extension', 'Entropy_Afterpath'
]

In [19]:
X = df_cleaned[X_col]
y = df_cleaned['URL_Type_obf_Type']

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay

In [21]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [23]:
model = DecisionTreeClassifier(
    random_state=42
)
model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [24]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [25]:
print("📌 [Confusion Matrix]")
print(confusion_matrix(y_test, y_pred))
print("\n📌 [Classification Report]")
print(classification_report(y_test, y_pred))

📌 [Confusion Matrix]
[[303   3   3   1]
 [  3 449   1   0]
 [  5   9 127  10]
 [  1   6   3 842]]

📌 [Classification Report]
              precision    recall  f1-score   support

  Defacement       0.97      0.98      0.97       310
     malware       0.96      0.99      0.98       453
    phishing       0.95      0.84      0.89       151
        spam       0.99      0.99      0.99       852

    accuracy                           0.97      1766
   macro avg       0.97      0.95      0.96      1766
weighted avg       0.97      0.97      0.97      1766



In [26]:
model = DecisionTreeClassifier(
    max_depth=20,    
    random_state=42
)
model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=20, random_state=42)

In [27]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [28]:
print("📌 [Confusion Matrix]")
print(confusion_matrix(y_test, y_pred))
print("\n📌 [Classification Report]")
print(classification_report(y_test, y_pred))

📌 [Confusion Matrix]
[[302   3   4   1]
 [  2 449   2   0]
 [  4   9 128  10]
 [  1   6   3 842]]

📌 [Classification Report]
              precision    recall  f1-score   support

  Defacement       0.98      0.97      0.98       310
     malware       0.96      0.99      0.98       453
    phishing       0.93      0.85      0.89       151
        spam       0.99      0.99      0.99       852

    accuracy                           0.97      1766
   macro avg       0.97      0.95      0.96      1766
weighted avg       0.97      0.97      0.97      1766



In [80]:
model = DecisionTreeClassifier(
    max_depth=19,    
    random_state=42
)
model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=19, random_state=42)

In [82]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [84]:
print("📌 [Confusion Matrix]")
print(confusion_matrix(y_test, y_pred))
print("\n📌 [Classification Report]")
print(classification_report(y_test, y_pred))

📌 [Confusion Matrix]
[[302   3   3   2]
 [  3 449   1   0]
 [  4   9 128  10]
 [  1   6   3 842]]

📌 [Classification Report]
              precision    recall  f1-score   support

  Defacement       0.97      0.97      0.97       310
     malware       0.96      0.99      0.98       453
    phishing       0.95      0.85      0.90       151
        spam       0.99      0.99      0.99       852

    accuracy                           0.97      1766
   macro avg       0.97      0.95      0.96      1766
weighted avg       0.97      0.97      0.97      1766



In [87]:
model = DecisionTreeClassifier(
    max_depth=18,    
    random_state=42
)
model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=18, random_state=42)

In [89]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [91]:
print("📌 [Confusion Matrix]")
print(confusion_matrix(y_test, y_pred))
print("\n📌 [Classification Report]")
print(classification_report(y_test, y_pred))

📌 [Confusion Matrix]
[[303   3   3   1]
 [  3 449   1   0]
 [  4   9 127  11]
 [  1   6   3 842]]

📌 [Classification Report]
              precision    recall  f1-score   support

  Defacement       0.97      0.98      0.98       310
     malware       0.96      0.99      0.98       453
    phishing       0.95      0.84      0.89       151
        spam       0.99      0.99      0.99       852

    accuracy                           0.97      1766
   macro avg       0.97      0.95      0.96      1766
weighted avg       0.97      0.97      0.97      1766



### 랜덤포레스트

In [94]:
from sklearn.ensemble import RandomForestClassifier

rfm = RandomForestClassifier(random_state=42)
rfm.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [96]:
y_pred = rfm.predict(X_test)
y_prob = rfm.predict_proba(X_test)[:, 1]

In [98]:
print("📌 [Confusion Matrix]")
print(confusion_matrix(y_test, y_pred))
print("\n📌 [Classification Report]")
print(classification_report(y_test, y_pred))

📌 [Confusion Matrix]
[[306   1   1   2]
 [  2 449   1   1]
 [  9   7 128   7]
 [  1   2   2 847]]

📌 [Classification Report]
              precision    recall  f1-score   support

  Defacement       0.96      0.99      0.97       310
     malware       0.98      0.99      0.98       453
    phishing       0.97      0.85      0.90       151
        spam       0.99      0.99      0.99       852

    accuracy                           0.98      1766
   macro avg       0.97      0.96      0.96      1766
weighted avg       0.98      0.98      0.98      1766



In [100]:
from sklearn.ensemble import RandomForestClassifier

rfm = RandomForestClassifier(random_state=42,
                            max_depth=20)
rfm.fit(X_train, y_train)

RandomForestClassifier(max_depth=20, random_state=42)

In [101]:
y_pred = rfm.predict(X_test)
y_prob = rfm.predict_proba(X_test)[:, 1]

In [102]:
print("📌 [Confusion Matrix]")
print(confusion_matrix(y_test, y_pred))
print("\n📌 [Classification Report]")
print(classification_report(y_test, y_pred))

📌 [Confusion Matrix]
[[306   1   1   2]
 [  2 449   1   1]
 [  8   9 126   8]
 [  1   2   2 847]]

📌 [Classification Report]
              precision    recall  f1-score   support

  Defacement       0.97      0.99      0.98       310
     malware       0.97      0.99      0.98       453
    phishing       0.97      0.83      0.90       151
        spam       0.99      0.99      0.99       852

    accuracy                           0.98      1766
   macro avg       0.97      0.95      0.96      1766
weighted avg       0.98      0.98      0.98      1766



In [106]:
from sklearn.ensemble import RandomForestClassifier

rfm = RandomForestClassifier(random_state=42,
                            max_depth=15)
rfm.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, random_state=42)

In [107]:
y_pred = rfm.predict(X_test)
y_prob = rfm.predict_proba(X_test)[:, 1]

In [108]:
print("📌 [Confusion Matrix]")
print(confusion_matrix(y_test, y_pred))
print("\n📌 [Classification Report]")
print(classification_report(y_test, y_pred))

📌 [Confusion Matrix]
[[307   1   2   0]
 [  2 448   1   2]
 [  8   9 123  11]
 [  1   2   2 847]]

📌 [Classification Report]
              precision    recall  f1-score   support

  Defacement       0.97      0.99      0.98       310
     malware       0.97      0.99      0.98       453
    phishing       0.96      0.81      0.88       151
        spam       0.98      0.99      0.99       852

    accuracy                           0.98      1766
   macro avg       0.97      0.95      0.96      1766
weighted avg       0.98      0.98      0.98      1766



### >> 디시젼 트리와 랜덤 포레스트의 정확도가 크지 않음

## 이진 분류

In [115]:
# ▶ Step 1: spam vs. rest
y_train_step1 = (y_train == 'spam').astype(int)
y_test_step1 = (y_test == 'spam').astype(int)

In [117]:
clf1 = RandomForestClassifier(class_weight='balanced', random_state=42)
clf1.fit(X_train, y_train_step1)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [119]:
y_pred_step1 = clf1.predict(X_test)

In [121]:
# 예측 결과 spam이면 확정, 아니면 다음 단계로
final_pred = np.full_like(y_test, fill_value='unknown', dtype=object)
final_pred[y_pred_step1 == 1] = 'spam'

In [125]:
# ▶ Step 2: malware vs. rest (excluding spam)
mask_step2 = (y_train != 'spam')
X_train_step2 = X_train[mask_step2]
y_train_step2 = y_train[mask_step2]
y_train_step2_bin = (y_train_step2 == 'malware').astype(int)

In [127]:
mask_test_step2 = (final_pred == 'unknown')
X_test_step2 = X_test[mask_test_step2]
y_test_step2 = y_test[mask_test_step2]
y_test_step2_bin = (y_test_step2 == 'malware').astype(int)

In [129]:
clf2 = RandomForestClassifier(class_weight='balanced', random_state=42)
clf2.fit(X_train_step2, y_train_step2_bin)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [135]:
# Step 2 예측
y_pred_step2 = clf2.predict(X_test_step2)

# 'unknown'인 인덱스 추출 (=> Step 2에 사용된 인덱스)
step2_indices = np.where(final_pred == 'unknown')[0]

# 이 중 예측이 1 (malware)인 것만 선택
malware_indices = step2_indices[y_pred_step2 == 1]

# malware로 지정
final_pred[malware_indices] = 'malware'


In [137]:
# Step 3: Defacement vs phishing
mask_step3 = (y_train != 'spam') & (y_train != 'malware')
X_train_step3 = X_train[mask_step3]
y_train_step3 = y_train[mask_step3]
y_train_step3_bin = (y_train_step3 == 'Defacement').astype(int)

mask_test_step3 = (final_pred == 'unknown')
X_test_step3 = X_test[mask_test_step3]
y_test_step3 = y_test[mask_test_step3]
y_test_step3_bin = (y_test_step3 == 'Defacement').astype(int)

clf3 = RandomForestClassifier(class_weight='balanced', random_state=42)
clf3.fit(X_train_step3, y_train_step3_bin)

y_pred_step3 = clf3.predict(X_test_step3)

# 예측 결과 적용
step3_indices = np.where(final_pred == 'unknown')[0]
deface_indices = step3_indices[y_pred_step3 == 1]
phishing_indices = step3_indices[y_pred_step3 == 0]

final_pred[deface_indices] = 'Defacement'
final_pred[phishing_indices] = 'phishing'


In [138]:
# ▶ 전체 평가
print("✅ 최종 분류 결과")
print(classification_report(y_test, final_pred))

✅ 최종 분류 결과
              precision    recall  f1-score   support

  Defacement       0.95      0.99      0.97       310
     malware       0.97      0.99      0.98       453
    phishing       0.96      0.87      0.91       151
        spam       1.00      0.99      0.99       852

    accuracy                           0.98      1766
   macro avg       0.97      0.96      0.96      1766
weighted avg       0.98      0.98      0.98      1766

